In [4]:
from src.connection.clinets import *
import json
from IPython.display import display, Markdown
from sentence_transformers import SentenceTransformer

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
embedding_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 608.36it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
def get__embeddings(enhanced_query):
    query_embedding = embedding_model.encode([enhanced_query])
    vector = query_embedding.tolist()
    vector = vector[0]

    return vector

In [7]:
vector = get__embeddings("Drug dealing Sentences")

In [2]:
def get_embeddings(text, model):

    response = embed_client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

In [ ]:
def message_chat(message):
    chat_response = chat_client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": message}],
        temperature=1
        )
    response_text = chat_response.choices[0].message.content
    
    return  response_text  

In [ ]:
def message_chat(messages):
    chat_response = chat_client.chat.completions.create(
        model="gpt-4.1",
        messages= messages ,
        temperature=1
        )
    response_text = chat_response.choices[0].message.content
    
    return  response_text

In [11]:
print(message_chat("what is my history"))

I don’t have access to your personal history or past interactions, unless you provide them here in the conversation. Each session is independent, so if you’d like to share details or context, I can better assist you based on that information!


In [8]:
import weaviate

In [ ]:
def search_chunks(query_embedding):
    with weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=WEAVIATE_API_KEY,
) as client:
        

        # Step 2.2: Use this collection
        Euro_Laws = client.collections.use("Euro_Laws")

        # Step 2.3: Perform a vector search with NearVector
        response = Euro_Laws.query.near_vector(
            near_vector= query_embedding, 
            limit=5
        )

        # Retrive nerest chunks with metadata and put in list 
        retrived = []
        for obj in response.objects:
            retrived.append(json.dumps(obj.properties))

        # Get the celex id for nerest chunks 
        # needs to take a copy of the celex only ##############################
        celex_ids = []
        for meta in retrived:
            key_value = json.loads(meta)
            celex_ids.append(key_value.get("celex", ""))


    return retrived , celex_ids

In [11]:
with weaviate.connect_to_weaviate_cloud(
    cluster_url= WEAVIATE_URL,
    auth_credentials= WEAVIATE_API_KEY,
) as client:
    
    if client.is_connected():
        print("true")

true


In [12]:
retrivev , celex = search_chunks(vector)

In [15]:
retrivev[0]

'{"legal_basis": "12002M031; 12002M034", "subject_matter": "sources and branches of the law;  European Union law;  justice;  criminal law", "document_length": 13663, "status": "In Force", "eurovoc": "penal code; Community law - national law; criminal procedure; penalty; drug traffic", "chunk_number": 2, "authors": "European Council", "text": "when the offender has supplied the competent authorities with valuable information. (7) It is necessary to take measures to enable the confiscation of the proceeds of the offences referred to in this Framework Decision. (8) Measures should be taken to ensure that legal persons can be held liable for the criminal offences referred to by this Framework Decision which are committed for their benefit. (9) The effectiveness of the efforts made to tackle illicit drug trafficking depends essentially on the harmonisation of the national measures implementing this Framework Decision, HAS DECIDED AS FOLLOWS: Article 1 Definitions For the purposes of this Fr

In [14]:
celex

['32004F0757', '32004F0757', '32014D0688', '31997R2046', '32013R1382']

In [ ]:
def get_docs_celex(celex_ids):
    docs_list = []

    for celex_id in celex_ids:
        doc = laws[laws['CELEX'] == celex_id]['act_raw_text'].iloc[0]
        docs_list.append(doc)

    return docs_list  

In [ ]:
with weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=WEAVIATE_API_KEY
) as client:



        # Step 2.2: Use this collection
        Euro_Laws = client.collections.use("Euro_Laws")

        # Step 2.3: Perform a vector search with NearVector
        response = Euro_Laws.query.near_vector(
            near_vector= query_embedding, 
            limit=5
        )

        for obj in response.objects:
            print(json.dumps(obj.properties, indent=5)) 

{
     "celex": "31994L0033",
     "legal_basis": "11992E118A; 11992E189C",
     "eurovoc": "child labour; occupational hazard; time worked; child abuse; work for young people; employment standard",
     "status": "In Force",
     "total_chunks": 12,
     "text": "20.8.1994 EN Official Journal of the European Communities L 216/12 COUNCIL DIRECTIVE 94/33/EC of 22 June 1994 on the protection of young people at work THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty establishing the European Community, and in particular Article 118a thereof, Having regard to the proposal from the Commission (1), Having regard to the opinion of the Economic and Social Committee (2), Acting in accordance with the procedure referred to in Article 189c of the Treaty (3), Whereas Article 118a of the Treaty provides that the Council shall adopt, by means of directives, minimum requirements to encourage improvements, especially in the working environment, as regards the health and safety of workers; 

In [15]:
doc = """

20.8.1994 EN Official Journal of the European Communities L 216/12 COUNCIL DIRECTIVE 94/33/EC of 22 June 1994 on the protection of young people at work THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty establishing the European Community, and in particular Article 118a thereof, Having regard to the proposal from the Commission (1), Having regard to the opinion of the Economic and Social Committee (2), Acting in accordance with the procedure referred to in Article 189c of the Treaty (3), Whereas Article 118a of the Treaty provides that the Council shall adopt, by means of directives, minimum requirements to encourage improvements, especially in the working environment, as regards the health and safety of workers; Whereas, under that Article, such directives must avoid imposing administrative, financial and legal constraints in a way which would hold back the creation and development of small and medium-sized undertakings; Whereas points 20 and 22 of the Community Charter of the Fundamental Social Rights of Workers, adopted by the European Council in Strasbourg on 9 December 1989, state that: 20. Without prejudice to such rules as may be more favourable to young people, in particular those ensuring their preparation for work through vocational training, and subject to derogations limited to certain light work, the minimum employment age must not be lower than the minimum school-leaving age and, in any case, not lower than 15 years; 22. Appropriate measures must be taken to adjust labour regulations applicable to young workers so that their specific development and vocational training and access to employment needs are met.The duration of work must, in particular, be limited  without it being possible to circumvent this limitation through recourse to overtime  and night work prohibited in the case of workers of under eighteen years of age, save in the case of certain jobs laid down in national legislation or regulations.; Whereas account should be taken of the principles of the International Labour Organization regarding the protection of young people at work, including those relating to the minimum age for access to employment or work; Whereas, in this Resolution on child labour (4), the European Parliament summarized the various aspects of work by young people and stressed its effects on their health, safety and physical and intellectual development, and pointed to the need to adopt a Directive harmonizing national legislation in the field; Whereas Article 15 of Council Directive 89/391/EEC of 12 June 1989 on the introduction of measures to encourage improvements in the safety and health of workers at work (5) provides that particularly sensitive risk groups must be protected against the dangers which specifically affect them; Whereas children and adolescents must be considered specific risk groups, and measures must be taken with regard to their safety and health; Whereas the vulnerability of children calls for Member States to prohibit their employment and ensure that the minimum working or employment age is not lower than the minimum age at which compulsory schooling as imposed by national law ends or 15 years in any event; whereas derogations from the prohibition on child labour may be admitted only in special cases and under the conditions stipulated in this Directive; whereas, under no circumstances, may such derogations be detrimental to regular school attendance or prevent children benefiting fully from their education; Whereas, in view of the nature of the transition from childhood to adult life, work by adolescents should be strictly regulated and protected; Whereas every employer should guarantee young people working conditions appropriate to their age; Whereas employers should implement the measures necessary to protect the safety and health of young people on the basis on an assessment of work-related hazards to the young; Whereas Member States should protect young people against any specific risks arising from their lack of experience, absence of awareness of existing or potential risks, or from their immaturity; Whereas Member States should therefore prohibit the employment of young people for the work specified by this Directive; Whereas the adoption of specific minimal requirements in respect of the organization of working time is likely to improve working conditions for young people; Whereas the maximum working time of young people should be strictly limited and night work by young people should be prohibited, with the exception of certain jobs specified by national legislation or rules; Whereas Member States should take the appropriate measures to ensure that the working time of adolescents receiving school education does not adversely affect their ability to benefit from that education; Whereas time spent on training by young persons working under a theoretical and/or practical combined work/training scheme or an in-plant work-experience should be counted as working time; Whereas, in order to ensure the safety and health of young people, the latter should be granted minimum daily, weekly and annual periods of rest and adequate breaks; Whereas, with respect to the weekly rest period, due account should be taken of the diversity of cultural, ethnic, religious and other factors prevailing in the Member States; whereas in particular, it is ultimately for each Member State to decide whether Sunday should be included in the weekly rest period, and if so to what extent; Whereas appropriate work experience may contribute to the aim of preparing young people for adult working and social life, provided it is ensured that any harm to their safety, health and development is avoided; Whereas, although derogations from the bans and limitations imposed by this Directive would appear indispensable for certain activities or particular situations, applications thereof must not prejudice the principles underlying the established protection system; Whereas this Directive constitutes a tangible step towards developing the social dimension of the internal market; Whereas the application in practice of the system of protection laid down by this Directive will require that Member States implement a system of effective and proportionate measures; Whereas the implementation of some provisions of this Directive poses particular problems for one Member State with regard to its system of protection for young people at work; whereas that Member State should therefore be allowed to refrain from implementing the relevant provisions for a suitable period, HAS ADOPTED THIS DIRECTIVE: SECTION I Article 1 Purpose 1. Member States shall take the necessary measures to prohibit work by children. They shall ensure, under the conditions laid down by this Directive, that the minimum working or employment age is not lower than the minimum age at which compulsory full-time schooling as imposed by national law ends or 15 years in any event. 2. Member States ensure that work by adolescents is strictly regulated and protected under the conditions laid down in this Directive. 3. Member States shall ensure in general that employers guarantee that young people have working conditions which suit their age. They shall ensure that young people are protected against economic exploitation and against any work likely to harm their safety, health or physical, mental, moral or social development or to jeopardize their education. Article 2 Scope 1. This Directive shall apply to any person under 18 years of age having an employment contract or an employment relationship defined by the law in force in a Member State and/or governed by the law in force in a Member State. 2. Member States may make legislative or regulatory provision for this Directive not to apply, within the limits and under the conditions which they set by legislative or regulatory provision, to occasional work or short-term work involving: (a) domestic service in a privat household, or (b) work regarded as not being harmful, damaging or dangerous to young people in a family undertaking. Article 3 Definitons For the purposes of this Directive: (a) young person shall mean any person under 18 years of age referred to in Article 2 (1); (b) child shall mean any young person of less than 15 years of age or who is still subject to compulsory full-time schooling under national law; (c) adolescent shall mean any young person of at least 15 years of age but less than 18 years of age who is no longer subject to compulsory full-time schooling under national law; (d) light work shall mean all work which, on account of the inherent nature of the tasks which it involves and the particular conditions under which they are performed: (i) is not likely to be harmful to the safety, health or development of children, and (ii) is not such as to be harmful to their attendance at school, their participation in vocational guidance or training programmes approved by the competent authority or their capacity to benefit from the instruction received; (e) working time shall mean any period during which the young person is at work, at the employer's disposal and carrying out his activity or duties in accordance with national legislation and/or practice; (f) rest period shall mean any period which is not working time. Article 4 Prohibition of work by children 1. Member States shall adopt the measures necessary to prohibit work by children. 2. Taking into account the objectives set out in Article 1, Member States may make legislative or regulatory provision for the prohibition of work by children not to apply to: (a) children pursuing the activities set out in Article 5; (b) children of at least 14 years of age working under a combined work/training scheme or an in-plant work-experience scheme, provided that such work is done in accordance with the conditions laid down by the competent authority; (c) children of at least 14 years of age performing light work other than that covered by Article 5; light work other than that covered by Article 5 may, however, be performed by children of 13 years of age for a limited number of hours per week in the case of categories of work determined by national legislation. 3. Member States that make use of the opinion referred to in paragraph 2 (c) shall determine, subject to the provisions of this Directive, the working conditions relating to the light work in question. Article 5 Cultural or similar activities 1. The employment of children for the purposes of performance in cultural, artistic, sports or advertising activities shall be subject to prior authorization to be given by the competent authority in individual cases. 2. Member States shall by legislative or regulatory provision lay down the working conditions for children in the cases referred to in paragraph 1 and the details of the prior authorization procedure, on condition that the activities: (i) are not likely to be harmful to the safety, health or development of children, and (ii) are not such as to be harmful to their attendance at school, their participation in vocational guidance or training programmes approved by the competent authority or their capacity to benefit from the instruction received. 3. By way of derogation from the procedure laid down in paragraph 1, in the case of children of at least 13 years of age, Member States may authorize, by legislative or regulatory provision, in accordance with conditions which they shall determine, the employment of children for the purposes of performance in cultural, artistic, sports or advertising activities. 4. The Member States which have a specific authorization system for modelling agencies with regard to the activities of children may retain that system. SECTION II Article 6 General obligations on employers 1. Without prejudice to Article 4 (1), the employer shall adopt the measures necessary to protect the safety and health of young people, taking particular account of the specific risks referred to in Article 7 (1). 2. The employer shall implement the measures provided for in paragraph 1 on the basis of an assessment of the hazards to young people in connection with their work. The assessment must be made before young people begin work and when there is any major change in working conditions and must pay particular attention to the following points: (a) the fitting-out and layout of the workplace and the workstation; (b) the nature, degree and duration of exposure to physical, biological and chemical agents; (c) the form, range and use of work equipment, in particular agents, machines, apparatus and devices, and the way in which they are handled; (d) the arrangement of work processes and operations and the way in which these are combined (organization of work); (e) the level of training and instruction given to young people. Where this assessment shows that there is a risk to the safety, the physical or mental health or development of young people, an appropriate free assessment and monitoring of their health shall be provided at regular intervals without prejudice to Directive 89/391/EEC. The free health assessment and monitoring may form part of a national health system. 3. The employer shall inform young people of possible risks and of all measures adopted concerning their safety and health. Furthermore, he shall inform the legal representatives of children of possible risks and of all measures adopted concerning children's safety and health, 4. The employer shall involve the protective and preventive services referred to in Article 7 of Directive 89/391/EEC in the planning, implementation and monitoring of the safety and health conditions applicable to young people. Article 7 Vulnerability of young people  Prohibition of work 1. Member States shall ensure that young people are protected from any specific risks to their safety, health and development which are a consequence of their lack of experience, of absence of awareness of existing or potential risks or of the fact that young people have not yet fully matured. 2. Without prejudice to Article 4 (1), Member States shall to this end prohibit the employment of young people for: (a) work which is objectively beyond their phyiscal or psychological capacity; (b) work involving harmful exposure to agents which are toxic, carcinogenic, cause heritable genetic damage, or harm to the unborn child or which in any other way chronically affect human health; (c) work involving harmful exposure to radiation; (d) work involving the risk of accidents which it may be assumed cannot be recognized or avoided by young persons owing to their insufficient attention to safety or lack of experience or training; or (e) work in which there is a risk to health from extreme cold or heat, or from noise or vibration. Work which is likely to entail specific risks for young people within the meaning of paragraph 1 includes:  work involving harmful exposure to the physical, biological and chemical agents referred to in point I of the Annex, and  processes and work referred to in point II of the Annex. 3. Member States may, by legislative or regulatory provision, authorize derogations from paragraph 2 in the case of adolescents where such derogations are indispensable for their vocational training, provided that protection of their safety and health is ensured by the fact that the work is performed under the supervision of a competent person within the meaning of Article 7 of Directive 89/391/EEC and provided that the protection afforded by that Directive is guaranteed. SECTION III Article 8 Working time 1. Member States which make use of the option in Article 4 (2) (b) or (c) shall adopt the measures necessary to limit the working time of children to: (a) eight hours a day and 40 hours a week for work performed under a combined work/training scheme or an in-plant work-experience scheme; (b) two hours on a school day and 12 hours a week for work performed in term-time outside the hours fixed for school attendance, provided that this is not prohibited by national legislation and/or practice; in no circumstances may the daily working time exceed seven hours; this limit may be raised to eight hours in the case of children who have reached the age of 15; (c) seven hours a day and 35 hours a week for work performed during a period of at least a week when school is not operating; these limits may be raised to eight hours a day and 40 hours a week in the case of chidren who have reached the age of 15; (d) seven hours a day and 35 hours a week for light work performed by children no longer subject to compulsory full-time schooling under national law. 2. Member States shall adopt the measures necessary to limit the working time of adolescents to eight hours a day and 40 hours a week. 3. The time spent on training by a young person working under a theoretical and/or practical combined work/training scheme or an in-plant work-experience scheme shall be counted as working time. 4. Where a young person is employed by more than one employer, working days and working time shall be cumulative. 5. Member States may, by legislative or regulatory provision, authorize derogations from paragraph 1 (a) and paragraph 2 either by way of exception or where there are objective grounds for so doing. Member States shall, by legislative or regulatory provision, determine the conditions, limits and procedure for implementing such derogations. Article 9 Night work 1. (a) Member States which make use of the option in Article 4 (2) (b) or (c) shall adopt the measures necessary to prohibit work by children between 8 p.m. and 6 a.m. (b) Member States shall adopt the measures necessary to prohibit work by adolescents either between 10 p.m. and 6 a.m. or between 11 p.m. and 7 a.m. 2. (a) Member States may, by legislative or regulatory provision, authorize work by adolescents in specific areas of activity during the period in which night work is prohibited as referred to in paragraph 1 (b). In that event, Member States shall take appropriate measures to ensure that the adolescent is supervised by an adult where such supervision is necessary for the adolescent's protection. (b) If point (a) is applied, work shall continue to be prohibited between midnight and 4 a.m. However, Member States may, by legislative or regulatory provision, authorize work by adolescents during the period in which night work is prohibited in the following cases, where there are objective grounds for so doing and provided that adolescents are allowed suitable compensatory rest time and that the objectives set out in Article 1 are not called into question:  work performed in the shipping or fisheries sectors;  work performed in the context of the armed forces or the police;  work performed in hospitals or similar establishments;  cultural, artistic, sports or advertising activities. 3. Prior to any assignment to night work and at regular intervals thereafter, adolescents shall be entitled to a free assessment of their health and capacities, unless the work they do during the period during which work is prohibited is of an exceptional nature. Article 10 Rest period 1. (a) Member States which make use of the option in Article 4 (2) (b) or (c) shall adopt the measures necessary to ensure that, for each 24-hour period, children are entitled to a minimum rest period of 14 consecutive hours. (b) Member States shall adopt the measures necessary to ensure that, for each 24-hour period, adolescents are entitled to a minimum rest period of 12 consecutive hours. 2. Member States shall adopt the measures necessary to ensure that, for each seven-day period:  children in respect of whom they have made use of the option in Article 4 (2) (b) or (c), and  adolescents are entitled to a minimum rest period of two days, which shall be consecutive if possible. Where justified by technical or organization reasons, the minimum rest period may be reduced, but may in no circumstances be less than 36 consecutive hours. The minimum rest period referred to in the first and second subparagraphs shall in principle include Sunday. 3. Member States may, by legislative or regulatory provision, provide for the minimum rest periods referred to in pargraphs 1 and 2 to be interrupted in the case of activities involving periods of work that are split up over the day or are of short duration. 4. Member States may make legislative or regulatory provision for derogations from paragraph 1 (b) and paragraph 2 in respect of adolescents in the following cases, where there are objective grounds for so doing and provided that they are granted appropriate compensatory rest time and that the objetives set out in Article 1 are not called into question: (a) work performed in the shipping or fisheries sectors; (b) work performed in the context of the armed forces or the police; (c) work performed in hospitals or similar establishments; (d) work performed in agriculture; (e) work performed in the tourism industry or in the hotel, restaurant and cafe sector; (f) activities involving periods of work split up over the day. Article 11 Annual rest Member States which make use of the option referred to in Article 4 (2) (b) or (c) shall see to it that a period free of any work is included, as far as possible, in the school holidays of children subject to compulsory full-time schooling under national law. Article 12 Breaks Member States shall adopt the measures necessary to ensure that, where daily working time is more than four and a half hours, young people are entitled to a break of at least 30 minutes, which shall be consecutive if possible. Article 13 Work by adolescents in the event of force majeure Member States may, by legislative or regulatory provision, authorize derogations from Article 8 (2), Article 9 (1) (b), Article 10 (1) (b) and, in the case of adolescents, Article 12, for work in the circumstances referred to in Article 5 (4) of Directive 89/391/EEC, provided that such work is of a temporary nature and must be performed immediately, that adult workers are not available and that the adolescents are allowed equivalent compensatory rest time within the following three weeks. SECTION IV Article 14 Measures Each Member State shall lay down any necessary measures to be applied in the event of failure to comply with the provisions adopted in order to implement this Directive; such measures must be effective and proportionate. Article 15 Adaptation of the Annex Adaptations of a strictly technical nature to the Annex in the light of technical progress, changes in international rules or specifications and advances in knowledge in the field covered by this Directive shall be adopted in accordance with the procedure provided for in Article 17 of Directive 89/391/EEC. Article 16 Non-reducing clause Without prejudice to the right of Member States to develop, in the light of changing circumstances, different provisions on the protection of young people, as long as the minimum requirements provided for by this Directive are complied with, the implementation of this Directive shall not constitute valid grounds for reducing the general level of protection afforded to young people. Article 17 Final provisions 1. (a) Member States shall bring into force the laws, regulations and administrative provisions necessary to comply with this Directive not later than 22 June 1996 or ensure, by that date at the latest, that the two sides of industry introduce the requisite provisions by means of collective agreements, with Member States being required to make all the necessary provisions to enable them at all times to guarantee the results laid down by this Directive. (b) The United Kingdom may refrain from implementing the first subparagraph of Article 8 (1) (b) with regard to the provision relating to the maximum weekly working time, and also Article 8(2) and Article 9 (1) (b) and (2) for a period of four years from the date specified in subparagraph (a). The Commission shall submit a report on the effects of this provision. The Council, acting in accordance with the conditions laid down by the Treaty, shall decide whether this period should be extended. (c) Member States shall forthwith inform the Commission thereof. 2. When Member States adopt the measures referred to in paragraph 1, such measures shall contain a reference to this Directive or shall be accompanied by such reference on the occasion of their official publication. The methods of making such reference shall be laid down by Member States. 3. Member States shall communicate to the Commission the texts of the main provisions of national law which they have already adopted or adopt in the field governed by this Directive. 4. Member States shall report to the Commission every five years on the practical implementation of the provisions of this Directive, indicating the viewpoints of the two sides of industry. The Commission shall inform the European Parliament, the Council and the Economic and Social Committee thereof. 5. The Commission shall periodically submit to the European Parliament, the Council and the Economic and Social Committee a report on the application of this Directive taking into account pargraphs 1, 2, 3 and 4. Article 18 This Directive is addressed to the Member States. Done at Luxembourg, 22 June 1994. For the Council The President E. YIANNOPOULOS (1) OJ No C 84, 4. 4. 1992, p. 7. (2) OJ No C 313, 30. 11. 1992, p. 70. (3) Opinion of the European Parliament of 17 December 1992 (OJ No C 21, 25. 1. 1993, p. 167). Council Common Position of 23 November 1993 (not yet published in the Official Journal) and Decision of the European Parliament of 9 March 1994 (OJ No C 91, 28. 3. 1994, p. 89). (4) OJ No C 190, 20. 7. 1987, p. 44. (5) OJ No L 183, 29. 6. 1989, p. 1. ANNEX Non-exhaustive list of agents, processes and work (Article 7(2), second subparagraph) I. Agents 1. Physical agents (a) lonizing radiation; (b) Work in a high-pressure atmosphere, e. g. in pressurized containers, diving. 2. Biological agents (a) Biological agents belonging to groups 3 and 4 within the meaning of Article 2 (d) of Council Directive 90/679/EEC of 26 November 1990 on the protection of workers from risks related to exposure to biological agents at work (Seventh individual Directive within the meaning of Article 16(1) of Directive 89/391/EEC) (1). 3. Chemical agents (a) Substances and preparations classified according to Council Directive 67/548/EEC of 27 June 1967 on the approximation of laws, regulations and administrative provisions relating to the classification, packaging and labelling of dangerous substances (2) with amendments and Council Directive 88/379/EEC of 7 June 1988 on the approximation of the laws, regulations and administrative provisions of the Member States relating to the classification, packaging and labelling of dangerous preparations (3) as toxic (T), very toxic (Tx), corrosive (C) or explosive (E); (b) Substances and preparations classified according to Directives 67/548/EEC and 88/379/EEC as harmful (Xn) and with one or more of the following risk phrases:  danger of very serious irreversible effects (R39),  possible risk of irreversible effects (R40),  may cause sensitization by inhalation (R42),  may cause sensitization by skin contact (R43),  may cause cancer (R45),  may cause heritable genetic damage (R46),  danger of serious damage to health by prolonged exposure (R48),  may impair fertility (R60),  may cause harm to the unborn child (R61); (c) Substances and preparations classified according to Directives 67/548/EEC and 88/379/EEC as irritant (Xi) and with one or more of the following risk phrases:  highly flammable (R12);  may cause sensitization by inhalation (R42),  may cause sensitization by skin contact (R43), (d) Substances and preparations referred to Article 2 (c) of Council Directive 90/394/EEC of 28 June 1990 on the protection of workers from the risks related to exposure to carcinogens at work (Sixth individual Directive within the meaning of Article 16(1) of Directive 89/391/EEC; (4) (e) Lead and compounds thereof, inasmuch as the agents in question are absorbable by the human organism; (f) Asbestos. II. Processes and work 1. Processes at work referred to in Annex I to Directive 90/394/EEC. 2. Manufacture and handling of devices, fireworks or other objects containing explosives. 3. Work with fierce of poisonous animals. 4. Animal slaughtering on an industrial scale. 5. Work involving the handling of equipment for the production, storage or application of compressed, liquified or dissolved gases. 6. Work with vats, tanks, reservoirs or carboys containing chemical agents referred to in 1.3. 7. Work involving a risk of structural collapse. 8. Work involving high-voltage electrical hazards. 9. Work the pace of which is determined by machinery and involving payment by results. (1) OJ No L 374, 31. 12. 1990, p. 1. (2) OJ No 196, 16. 8. 1967, p. 1. Directive as last amended by Directive 93/679/EEC (OJ No L 268, 29. 10. 1993, p, 71). (3) OJ No L 187, 16. 7. 1988, p. 14. Directive as last amended by Directive 93/18/EEC (OJ No L 104, 29. 4. 1993, p. 46). (4) OJ No L 196, 26. 7. 1990, p. 1.

"""



In [16]:
text =f"""
Depend on this user question:
labour law age to work

and the retrived legal doc:

<doc>
{doc}
</doc>

Answer guide:
- read the doc well and use only the lines or laws related to user Question

"""

In [19]:
display(Markdown(message_chat(text)))

Based on Council Directive 94/33/EC of 22 June 1994 on the protection of young people at work (as referenced in the provided legal document), the EU-wide labour law minimum age to work is defined as follows:

**Minimum Employment Age:**  
- The minimum working or employment age must not be lower than the age at which compulsory full-time schooling ends under national law, **or 15 years in any event**.  
  - *Reference: Article 1(1); also reinforced in the Charter points 20 and 22 ("...the minimum employment age must not be lower than the minimum school-leaving age and, in any case, not lower than 15 years").*

**Definitions (Article 3):**  
- **Child**: a young person under 15, or still subject to compulsory full-time schooling.
- **Adolescent**: a young person aged at least 15 but under 18, who is no longer subject to compulsory full-time schooling.

**Exceptions:**  
- **Light work (Article 4(2)(c)):**  
  - Children of at least 14 may perform certain light work, under conditions set by national law.
  - Children of 13 may perform some light work for a limited number of hours per week, in categories of work determined by national law.
- **Cultural/Artistic/Sports/Advertising Activities (Article 5):**  
  - Children under 15 may be permitted to work in these fields with special prior authorisation and only if it does not harm health, development, or schooling.

**Overall:**
- **General rule**: No regular employment under age 15.
- **Possible exceptions**: Light work at 14 (sometimes 13), or specific authorised activities with protective conditions, and only if not detrimental to education, health, or development.

**Summary:**
> **The legal minimum age to work in the EU is generally 15 years old, or the end of compulsory schooling if this is higher. Exceptions exist for certain light work from age 14 (sometimes 13) and for special activities (like arts/sports/advertising) under specific safeguards.**

For details and exceptions, always check your country’s national law, as it may provide stricter rules or further clarifications.

In [22]:
answer = """

Based on Council Directive 94/33/EC of 22 June 1994 on the protection of young people at work (as referenced in the provided legal document), the EU-wide labour law minimum age to work is defined as follows:

**Minimum Employment Age:**  
- The minimum working or employment age must not be lower than the age at which compulsory full-time schooling ends under national law, **or 15 years in any event**.  
  - *Reference: Article 1(1); also reinforced in the Charter points 20 and 22 ("...the minimum employment age must not be lower than the minimum school-leaving age and, in any case, not lower than 15 years").*

**Definitions (Article 3):**  
- **Child**: a young person under 15, or still subject to compulsory full-time schooling.
- **Adolescent**: a young person aged at least 15 but under 18, who is no longer subject to compulsory full-time schooling.

**Exceptions:**  
- **Light work (Article 4(2)(c)):**  
  - Children of at least 14 may perform certain light work, under conditions set by national law.
  - Children of 13 may perform some light work for a limited number of hours per week, in categories of work determined by national law.
- **Cultural/Artistic/Sports/Advertising Activities (Article 5):**  
  - Children under 15 may be permitted to work in these fields with special prior authorisation and only if it does not harm health, development, or schooling.

**Overall:**
- **General rule**: No regular employment under age 15.
- **Possible exceptions**: Light work at 14 (sometimes 13), or specific authorised activities with protective conditions, and only if not detrimental to education, health, or development.

**Summary:**
> **The legal minimum age to work in the EU is generally 15 years old, or the end of compulsory schooling if this is higher. Exceptions exist for certain light work from age 14 (sometimes 13) and for special activities (like arts/sports/advertising) under specific safeguards.**

For details and exceptions, always check your country’s national law, as it may provide stricter rules or further clarifications.

"""



In [21]:
message_history = []

def add_message(message_history:list[dict[str,str]], user_message:str, chat_response:str):

    message_history.append({"role": "user", "content": user_message})
    message_history.append({"role": "assistant", "content": chat_response})

    return message_history



In [23]:
message_history = add_message (message_history, text, answer)

In [ ]:
def message_with_context(user_qusetion):

    asking_prompt =f"""
    Depend on this user question:
    
    {user_qusetion}

    and the retrived legal doc:

    <doc>
    </doc>

    Answer guide:
    - read the doc well and use only the lines or laws related to user Question
    - if no doc and the user askes about previous chat answer him
    """

    message_history.append({"role": "user", "content": asking_prompt})

    chat_response = message_chat(message_history)

    message_history.append({"role": "assistant", "content": chat_response})


    return display(Markdown(chat_response))


In [27]:
message_with_context("depend on the doc how many hours kids with 14 or 13 or 15 YOE is allowed to work")

There is no document provided for this question. However, based on the previous information from the Council Directive 94/33/EC that you referred to earlier:

- **Children aged 13 or 14**: May perform "light work" for a limited number of hours per week as defined by national law.
    - During school term: Maximum 2 hours on a school day, 12 hours a week (outside school hours). Daily working time must not exceed 7 hours.
    - During school holidays: Maximum 7 hours a day, 35 hours a week (for those under 15). For those who have reached 15, this can be 8 hours a day, 40 hours a week.

- **Children aged 15**: If they have completed compulsory schooling, treated as "adolescents."
    - Maximum 8 hours a day, 40 hours a week.

**Note:** These limits come from the previous document (Council Directive 94/33/EC) and actual permitted hours may be more restricted under your country's national law. Always check local laws for specifics.

In [28]:
message_history

[{'role': 'user',
  'content': "\nDepend on this user question:\nlabour law age to work\n\nand the retrived legal doc:\n\n<doc>\n\n\n20.8.1994 EN Official Journal of the European Communities L 216/12 COUNCIL DIRECTIVE 94/33/EC of 22 June 1994 on the protection of young people at work THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty establishing the European Community, and in particular Article 118a thereof, Having regard to the proposal from the Commission (1), Having regard to the opinion of the Economic and Social Committee (2), Acting in accordance with the procedure referred to in Article 189c of the Treaty (3), Whereas Article 118a of the Treaty provides that the Council shall adopt, by means of directives, minimum requirements to encourage improvements, especially in the working environment, as regards the health and safety of workers; Whereas, under that Article, such directives must avoid imposing administrative, financial and legal constraints in a way which woul